# Notebook 3 — Modèle BK 1 bloc (Erickson éq. 5, non-dimensionnel)

Système non-dimensionnel à 1 bloc (Erickson et al. 2011, éq. 5, *slip law*) :

$$\dot{\bar{u}} = \bar{v} - 1, \quad
  \dot{\bar{v}} = -\gamma^2\!\left[\bar{u} + \frac{1}{\xi}(\bar{f} + \bar{\Theta} + \ln\bar{v})\right], \quad
  \dot{\bar{\Theta}} = -\bar{v}\left[\bar{\Theta} + (1+\varepsilon)\ln\bar{v}\right]$$

Paramètres : γ = 0.5, ξ = 0.5, f̄ = 3.2.

**Contenu :**
1. **Sweep en ε** : séries temporelles (ū, v̄, Θ̄) pour ε ∈ {0.02, 0.5, 2, 8, 12, 20}
2. **Diagramme de bifurcation** : extrema de v̄ en régime permanent vs ε


## 1 — Sweep en ε : séries temporelles

In [2]:
"""
Epsilon sweep — non-dimensional formulation of Erickson et al. (2011) eq.(5)
Remplace la version dimensionnelle (Scholz) qui produisait des oscillations
artefactuelles liées au découplage stick/slip discret.

Système non-dimensional (slip law, 1 bloc) :
    ū'   = v̄ - 1
    v̄'   = -γ² · [ū + (1/ξ)·(f̄ + Θ̄ + ln(v̄))]
    Θ̄'   = -v̄ · [Θ̄ + (1+ε)·ln(v̄)]

with  γ = 0.5,  ξ = 0.5,  f̄ = 3.2
CI :  ū(0) = ū_eq + 0.01,  v̄(0) = 1,  Θ̄(0) = 0
      ū_eq = -f̄/ξ = -6.4   (point fixe)

La variable v̄ est la vitesse du bloc RELATIVE à la plaque, normalisée par v₀.
v̄ = 1  → le bloc glisse à la même vitesse que la plaque (regime stationnaire).
v̄ > 1  → glissement rapide (slip).
v̄ < 1  → glissement lent (quasi-sticking).

ε = (b - a) / a  paramètre de velocity-weakening :
    ε < ε_crit ≈ 0.166 : stable regime (point fixe)
    ε > ε_crit         : limit cycle / chaos
"""

import numpy as np
from scipy.integrate import solve_ivp
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import gc

# ── System parameters ────────────────────────────────────────────────────
GAMMA = 0.5
XI    = 0.5
F_BAR = 3.2
U_EQ  = -F_BAR / XI      # = -6.4  (equilibrium)
EPS_CRIT = 0.166          # bifurcation de Hopf analytique

# ── Conditions initiales ─────────────────────────────────────────────────────
Y0 = [U_EQ + 0.01, 1.0, 0.0]   # petit écart à l'equilibrium

# ── RHS ──────────────────────────────────────────────────────────────────────
def make_rhs(eps):
    def rhs(t, y):
        u, v, Th = y
        v  = max(v, 1e-10)          # v̄ > 0  (slip law : glissement forward)
        lv = np.log(v)
        du  = v - 1.0
        dv  = -GAMMA**2 * (u + (1.0 / XI) * (F_BAR + Th + lv))
        dTh = -v * (Th + (1.0 + eps) * lv)
        return [du, dv, dTh]
    return rhs

# ── Durées de simulation adaptées par regime ─────────────────────────────────
def t_end_for(eps):
    """Adaptive simulation duration: long enough to see steady state,
    assez courte pour rester raisonnable en temps de calcul.
    NOTE : ε=4 est exclu — c'est le cas maximalement raide (resticking brutal
    after le pic de vitesse, requiert plusieurs heures de calcul)."""
    if eps < 0.1:   return 400
    if eps < 0.5:   return 600
    if eps < 2.0:   return 300
    if eps < 5.0:   return 300    # eps=3 est OK ici
    if eps < 9.0:   return 300
    return 3000

# ── Simulation d'un cas ───────────────────────────────────────────────────────
def simulate(eps):
    t_end = t_end_for(eps)
    sol = solve_ivp(
        make_rhs(eps),
        [0, t_end], Y0,
        method='Radau',
        rtol=1e-5, atol=1e-7,
        max_step=2.0,
    )
    return sol.t, sol.y[0], sol.y[1], sol.y[2]   # t, ū, v̄, Θ̄

# ── Figure : sweep en epsilon ─────────────────────────────────────────────────
def plot_epsilon_sweep(eps_list):
    n = len(eps_list)
    colors = plt.cm.plasma(np.linspace(0.1, 0.9, n))

    fig, axes = plt.subplots(n, 3, figsize=(15, 3 * n))
    fig.suptitle(
        'Sweep en ε — formulation non-dimensional Erickson (2011)\n'
        r'$\gamma=0.5,\ \xi=0.5,\ \bar{f}=3.2$  —  '
        r'$\varepsilon_{crit}^{Hopf} \approx 0.166$',
        fontsize=11, fontweight='bold'
    )

    axes[0, 0].set_title(r'Position relative $\bar{u}$',      fontsize=10)
    axes[0, 1].set_title(r'Vitesse relative $\bar{v}$',       fontsize=10)
    axes[0, 2].set_title(r'Variable d\'état $\bar{\Theta}$',  fontsize=10)

    for row, (eps, col) in enumerate(zip(eps_list, colors)):
        print(f'  ε = {eps:5.2f} ...', flush=True, end=' ')
        t, u, v, Th = simulate(eps)

        # Discard transient (half de la simulation)
        i0 = len(t) // 2

        regime = 'stable' if eps < EPS_CRIT else (
                 'periodique' if eps < 2.0 else 'quasi-chaotic / divergent')
        label_y = rf'$\varepsilon={eps}$' + '\n' + regime

        axes[row, 0].plot(t[i0:], u[i0:], color=col, lw=0.8)
        axes[row, 0].axhline(U_EQ, color='gray', lw=0.6, ls='--', alpha=0.5)
        axes[row, 0].set_ylabel(label_y, fontsize=8, rotation=0,
                                labelpad=90, va='center')
        axes[row, 0].grid(True, alpha=0.25)

        axes[row, 1].plot(t[i0:], v[i0:], color=col, lw=0.8)
        axes[row, 1].axhline(1.0, color='gray', lw=0.6, ls='--', alpha=0.5,
                             label='v̄=1 (stationnaire)')
        axes[row, 1].grid(True, alpha=0.25)

        axes[row, 2].plot(t[i0:], Th[i0:], color=col, lw=0.8)
        axes[row, 2].axhline(0.0, color='gray', lw=0.6, ls='--', alpha=0.5)
        axes[row, 2].grid(True, alpha=0.25)

        if row == n - 1:
            for c in range(3):
                axes[row, c].set_xlabel(r'$\bar{t}$', fontsize=9)

        # Afficher vmax dans le titre de la colonne vitesse
        v_perm = v[i0:]
        axes[row, 1].set_title(
            f'|v̄|_max = {np.abs(v_perm).max():.2e}', fontsize=7, pad=1)

        print(f'|v̄|_max = {np.abs(v_perm).max():.2e}')
        del t, u, v, Th
        gc.collect()

    # Colorbar
    sm = plt.cm.ScalarMappable(cmap='plasma',
                               norm=plt.Normalize(min(eps_list), max(eps_list)))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes[:, 2], orientation='vertical',
                        fraction=0.04, pad=0.02)
    cbar.set_label(r'$\varepsilon$', fontsize=10)
    cbar.set_ticks(eps_list)

    plt.tight_layout()
    plt.savefig('scholz_epsilon_sweep_corrected.png', dpi=140, bbox_inches='tight')
    plt.close(fig)
    print('\nFigure saved → scholz_epsilon_sweep_corrected.png')
    gc.collect()


# ── Main ──────────────────────────────────────────────────────────────────────
if __name__ == '__main__':
    eps_list = [0.02, 0.5, 1.0, 2.0, 3.0, 8.0, 12.0, 20.0]
    # NOTE: ε=4 est omis — cas maximalement raide, coût de calcul prohibitif.
    print(f'Sweep sur {len(eps_list)} valeurs de ε...\n')
    plot_epsilon_sweep(eps_list)

Sweep sur 8 valeurs de ε...

  ε =  0.02 ... |v̄|_max = 1.00e+00
  ε =  0.50 ... |v̄|_max = 2.56e+00
  ε =  1.00 ... |v̄|_max = 4.11e+00
  ε =  2.00 ... |v̄|_max = 8.22e+00
  ε =  3.00 ... |v̄|_max = 1.35e+01
  ε =  8.00 ... |v̄|_max = 5.71e+02
  ε = 12.00 ... |v̄|_max = 3.05e+03
  ε = 20.00 ... |v̄|_max = 4.19e+04


C:\Users\coren\AppData\Local\Temp\ipykernel_30776\1460578592.py:144: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()



Figure sauvegardée → scholz_epsilon_sweep_corrected.png


---
## 2 — Diagramme de bifurcation

Pour chaque ε, on intègre jusqu'à T = 1500 (Radau implicite), puis on collecte les extrema 
locaux de v̄ dans le dernier quart. Un seul extremum → cycle limite. Nuage → chaos.


In [3]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import time


In [4]:
# ── Parameters fixes ──────────────────────────────────────────────────────
GAMMA = 0.5
XI    = 0.5
F_BAR = 3.2
U_EQ  = -F_BAR / XI   # u_bar au point fixe = -6.4

def rhs(t, y, eps):
    """
    Right-hand side of eq.(5) of Erickson et al. (2011).
    y = [u_bar, v_bar, Theta_bar]
    """
    u, v, Th = y
    v   = max(v, 1e-10)        # v_bar > 0 (slip law : glissement forward)
    lnv = np.log(v)
    du  = v - 1.0
    dv  = -GAMMA**2 * (u + (1.0/XI)*(F_BAR + Th + lnv))
    dTh = -v * (Th + (1.0 + eps)*lnv)
    return [du, dv, dTh]

y0 = [U_EQ + 0.01, 1.0, 0.0]   # CI : léger écart à l'equilibrium

In [5]:
eps_bifurc = np.concatenate([
    np.linspace(0.10, 0.40, 40),   # autour de la bifurcation de Hopf
    np.linspace(0.40, 2.00, 25),   # limit cycle / grandes amplitudes
])

bif_eps = []
bif_v   = []

print(f'Calcul du diagramme ({len(eps_bifurc)} valeurs d eps)...')
t0 = time.time()

for eps in eps_bifurc:
    sol = solve_ivp(
        lambda t, y: rhs(t, y, eps),
        [0, 1500], y0,
        method='Radau', rtol=1e-5, atol=1e-7, max_step=5.0
    )
    v = sol.y[1]; t = sol.t
    mask = t > 0.75 * t[-1]          # dernier quart = steady state
    v_ss = v[mask]
    # Extrema locaux (pics)
    pks = [v_ss[i] for i in range(1, len(v_ss)-1)
           if v_ss[i] > v_ss[i-1] and v_ss[i] > v_ss[i+1] and v_ss[i] > 1.01]
    for vp in pks[-25:]:             # garder les 25 derniers pics
        bif_eps.append(eps)
        bif_v.append(vp)

print(f'Fait en {time.time()-t0:.1f}s')

Calcul du diagramme (65 values d eps)...
Fait en 48.8s


In [6]:
# ── Time series pour 4 valeurs représentatives de eps ──────────────
eps_plot = [0.02, 0.20, 0.50, 2.0, 12.0, 20.0]
T_plot   = [300,  600,  300,  300, 600, 600]
labels   = {
    0.02: 'ε=0.02',
    0.20: 'ε=0.20',
    0.50: 'ε=0.50',
    2.0:  'ε=2.0',
    4.0: 'ε=4.0',
    8.0: 'ε=8.0',
    12.0: 'ε=12.0',
    20.0: 'ε=20.0'
}

sims = {}
for eps, T in zip(eps_plot, T_plot):
    sol = solve_ivp(
        lambda t, y, e=eps: rhs(t, y, e),
        [0, T], y0,
        method='Radau', rtol=1e-5, atol=1e-7, max_step=2.0
    )
    sims[eps] = (sol.t, sol.y[1])
    print(f'eps={eps}: {len(sol.t)} points')

eps=0.02: 240 points
eps=0.2: 1075 points
eps=0.5: 842 points
eps=2.0: 940 points
eps=12.0: 605 points
eps=20.0: 489 points


In [7]:
fig = plt.figure(figsize=(13, 9))
gs  = fig.add_gridspec(3, 6, hspace=0.5, wspace=0.35)

# ── Bifurcation diagram (2 lignes du haut) ────────────────────────────
ax_bif = fig.add_subplot(gs[0:2, :])
ax_bif.scatter(bif_eps, bif_v, s=1.2, c='#1D9E75', alpha=0.6, rasterized=True)
ax_bif.axvline(0.166, color='#D85A30', lw=1.8, ls='--',
               label=r'$\varepsilon_{crit}$(Hopf) = 0.166')
ax_bif.set_xlabel('ε', fontsize=12)
ax_bif.set_ylabel('$\\bar{v}_{max}$ en steady state', fontsize=11)
ax_bif.set_title(
    'Bifurcation diagram — extremal values of $\\bar{v}$ en steady state\n'
    r'$\gamma=0.5,\;\xi=0.5,\;\bar{f}=3.2$',
    fontsize=10, fontweight='bold')
ax_bif.legend(fontsize=9)
ax_bif.grid(True, ls=':', alpha=0.4)

# ── Time series (ligne du bas) ──────────────────────────────────────
for col, eps in enumerate(eps_plot):
    ax = fig.add_subplot(gs[2, col])
    ts, vs = sims[eps]
    i_ss = np.searchsorted(ts, 0.5*ts[-1])   # ignorer le transitoire
    ax.plot(ts[i_ss:], vs[i_ss:], lw=0.7, color='#378ADD', rasterized=True)
    ax.axhline(1.0, color='gray', lw=0.7, ls='--', alpha=0.6)
    ax.set_title(labels[eps], fontsize=8.5)
    ax.set_xlabel('$\\bar{t}$', fontsize=8)
    if col == 0:
        ax.set_ylabel('$\\bar{v}$', fontsize=9)
    ax.grid(True, ls=':', alpha=0.4)

plt.savefig('erickson_bifurcation.png', dpi=140, bbox_inches='tight')
plt.show()
print('Saved: erickson_bifurcation.png')

Sauvegardé : erickson_bifurcation.png


C:\Users\coren\AppData\Local\Temp\ipykernel_30776\3436872699.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
